In [1]:
import requests

def fetch_html(url, headers=None, timeout=10):
    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()
    return response.text


In [2]:
# Test 0: quotes.toscrape.com — basic fetch, expect title in HTML

url0 = "https://quotes.toscrape.com/"
html0 = fetch_html(url0, headers={"User-Agent": "test-agent"}, timeout=5)
print("Test 0:\n", html0[:209])


Test 0:
 <!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">


In [14]:
# Test 1: quote.toscrape.com — Non-existent URL
url1 = "https://quote.toscrape.com/"
try:
    html1 = fetch_html(url1)
except Exception as e:
    html1 = e

print("Test 1:\n", str(html1))

Test 1:
 404 Client Error: Not Found for url: https://quote.toscrape.com/


In [15]:
# Test 2: reddit.com — compare content length with and without User-Agent
url2 = "https://www.google.com/"
try:
    html2 = fetch_html(url2)
    html_default = len(html2)
except Exception as e:
    html_default = e

try:
    headers2 = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/115.0.0.0 Safari/537.36"
        )
    }
    html2 = fetch_html(url2, headers=headers2)
    html_browser = len(html2)
except Exception as e:
    html_browser = e

print(f"Test 2:\n {str(html_default)}\n {str(html_browser)}")

Test 2:
 81017
 232250


In [16]:
# Test 3: gamefaqs.com — blocked without UA, allowed with UA
url3 = "https://wikipedia.org"
try:
    html3 = fetch_html(url3)
    html_default = len(html3)
except Exception as e:
    html_default = e

try:
    headers3 = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/115.0.0.0 Safari/537.36"
        )
    }
    html3 = fetch_html(url3, headers=headers3)
    html_good = len(html3)
except Exception as e:
    html_good = e

print(f"Test 3:\n {str(html_default)}\n {str(html_good)}")

Test 3:
 403 Client Error: Forbidden for url: https://wikipedia.org/
 120361


In [6]:
html = fetch_html("https://quotes.toscrape.com/")
print(html[:2000])

<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div class="col-md-4">
                <p>
                
                    <a href="/login">Login</a>
                
                </p>
            </div>
        </div>
    

<div class="row">
    <div class="col-md-8">

    <div class="quote" itemscope itemtype="http://schema.org/CreativeWork">
        <span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
        <span>by <small class="auth

In [4]:
from bs4 import BeautifulSoup

def scrape_basic(url):
    html = fetch_html(url)

    soup = BeautifulSoup(html, "html.parser")
    quotes =[]

    for quote_div in soup.find_all("div", class_="quote"):
        text = quote_div.find("span", class_='text').get_text()
        author = quote_div.find("small", class_="author").get_text()
        tag_links = quote_div.find_all("a", class_="tag")
        tags = [tag.get_text() for tag in tag_links]
        quotes.append({
            "text": text,
            "author": author,
            "tags": tags
        })
    return quotes

In [7]:
import json

data = scrape_basic("https://quotes.toscrape.com/")
# print(json.dumps(data[:2], indent=2))
print(json.dumps(data[:2], indent=2, ensure_ascii=False))


[
  {
    "text": "“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”",
    "author": "Albert Einstein",
    "tags": [
      "change",
      "deep-thoughts",
      "thinking",
      "world"
    ]
  },
  {
    "text": "“It is our choices, Harry, that show what we truly are, far more than our abilities.”",
    "author": "J.K. Rowling",
    "tags": [
      "abilities",
      "choices"
    ]
  }
]


In [8]:
import time
from urllib import parse

def scrape_paginated(base_url):
    all_quotes = []
    current_url = base_url

    while current_url is not None:
        html = fetch_html(current_url)
        soup = BeautifulSoup(html, "html.parser")

        for quote_div in soup.find_all("div", class_="quote"):
            text = quote_div.find("span", class_="text").get_text()
            author = quote_div.find("small", class_="author").get_text()
            tag_links = quote_div.find("div", class_="tags").find_all(
                "a", class_="tag"
            )
            tags = [tag.get_text() for tag in tag_links]

            all_quotes.append({
                "text": text,
                "author": author,
                "tags": tags,
            })

        next_li = soup.find("li", class_="next")

        if next_li is None:
            current_url = None
        else:
            next_href = next_li.find("a")["href"]
            current_url = parse.urljoin(current_url, next_href)
            time.sleep(1)

    return all_quotes


In [9]:
data = scrape_paginated("https://quotes.toscrape.com/")
print(len(data))
print(data[0])

100
{'text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'author': 'Albert Einstein', 'tags': ['change', 'deep-thoughts', 'thinking', 'world']}


In [16]:
url = "https://quotes.toscrape.com/"
try:
    data = scrape_paginated(url)
    if data:
        output = f"{data[0]}"
    else:
        output = "No quotes found."
    print(output)
except Exception as e:
    print(str(e))

{'text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'author': 'Albert Einstein', 'tags': ['change', 'deep-thoughts', 'thinking', 'world']}


In [10]:
import json

_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/115.0.0.0 Safari/537.36"
    )
}

def scrape_via_api(base_url):
    all_quotes = []
    page = 1
    while True:
        url = f"{base_url}/api/quotes?page={page}"
        data = json.loads(fetch_html(url, headers=_HEADERS))
        for quote in data["quotes"]:
            all_quotes.append({
                "text": quote["text"],
                "author": quote["author"]["name"],
                "tags": quote["tags"],
            })
        if not data["has_next"]:
            break
        page += 1
    return all_quotes

In [11]:
base = "https://quotes.toscrape.com"

quotes = scrape_via_api(base)
try:
    quotes = scrape_via_api(base)
    output = ""
    for i, q in enumerate(quotes):
        output += f"Quote #{i}:\n{q}\n"
    print(output)
except Exception as e:
    print(str(e))

Quote #0:
{'text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'author': 'Albert Einstein', 'tags': ['change', 'deep-thoughts', 'thinking', 'world']}
Quote #1:
{'text': '“It is our choices, Harry, that show what we truly are, far more than our abilities.”', 'author': 'J.K. Rowling', 'tags': ['abilities', 'choices']}
Quote #2:
{'text': '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”', 'author': 'Albert Einstein', 'tags': ['inspirational', 'life', 'live', 'miracle', 'miracles']}
Quote #3:
{'text': '“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”', 'author': 'Jane Austen', 'tags': ['aliteracy', 'books', 'classic', 'humor']}
Quote #4:
{'text': "“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”", 'author': 'Marilyn M

In [12]:
import json
from bs4 import BeautifulSoup

def _normalize_tags(keywords):
    if keywords is None:
        return []
    if isinstance(keywords, list):
        return keywords
    return [tag.strip() for tag in keywords.split(",")]

def extract_jsonld(url):
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")
    quotes = []
    scripts = soup.find_all("script", type="application/ld+json")
    for script in scripts:
        try:
            payload = json.loads(script.string)
        except (json.JSONDecodeError, TypeError):
            continue
        entries = payload if isinstance(payload, list) else [payload]
        for entry in entries:
            if entry.get("@type") != "Quote":
                continue
            quotes.append({
                "text": entry.get("text"),
                "author": entry.get("author", {}).get("name"),
                "tags": _normalize_tags(entry.get("keywords")),
            })
    return quotes

In [14]:
# Test 2: A dummy HTML page containing JSON-LD
def fake_fetch(url, headers=None, timeout=10):
    return '''
    <html><head><script type="application/ld+json">
    {
        "@context": "https://schema.org",
        "@type": "Quote",
        "text": "A witty quote",
        "author": {"@type": "Person","name": "Tester"},
        "keywords": ["fun","test"]
    }
    </script></head><body></body></html>
    '''

original_fetch = fetch_html
fetch_html = fake_fetch

try:
    data = extract_jsonld("dummy-url")
    print(data[0])
finally:
    fetch_html = original_fetch

{'text': 'A witty quote', 'author': 'Tester', 'tags': ['fun', 'test']}


In [15]:
def login_and_scrape(login_url, user, pwd):
    session = requests.Session()
    login_page = session.get(login_url)
    login_page.raise_for_status()
    soup = BeautifulSoup(login_page.text, "html.parser")
    csrf_token = soup.find("input", attrs={"name": "csrf_token"})["value"]
    payload = {"username": user, "password": pwd, "csrf_token": csrf_token}
    response = session.post(login_url, data=payload)
    response.raise_for_status()
    quotes_page = session.get("https://quotes.toscrape.com/")
    quotes_page.raise_for_status()
    soup = BeautifulSoup(quotes_page.text, "html.parser")
    quotes = []
    for quote_div in soup.find_all("div", class_="quote"):
        text = quote_div.find("span", class_="text").get_text()
        author = quote_div.find("small", class_="author").get_text()
        tag_links = quote_div.find("div", class_="tags").find_all("a", class_="tag")
        tags = [tag.get_text() for tag in tag_links]
        quotes.append({"text": text, "author": author, "tags": tags})
    return quotes

In [17]:

login_url = "https://quotes.toscrape.com/login"
try:
    quotes = login_and_scrape(login_url, user="foo", pwd="bar")
    output = ""
    for i, q in enumerate(quotes):
        output += f"Quote #{i}:\n{q}\n"
    print(output)
except Exception as e:
    print(str(e))


Quote #0:
{'text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'author': 'Albert Einstein', 'tags': ['change', 'deep-thoughts', 'thinking', 'world']}
Quote #1:
{'text': '“It is our choices, Harry, that show what we truly are, far more than our abilities.”', 'author': 'J.K. Rowling', 'tags': ['abilities', 'choices']}
Quote #2:
{'text': '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”', 'author': 'Albert Einstein', 'tags': ['inspirational', 'life', 'live', 'miracle', 'miracles']}
Quote #3:
{'text': '“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”', 'author': 'Jane Austen', 'tags': ['aliteracy', 'books', 'classic', 'humor']}
Quote #4:
{'text': "“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”", 'author': 'Marilyn M

In [13]:
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y -qq /tmp/chrome.deb
!pip install -U selenium --quiet

The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 96 not upgraded.
Need to get 11.2 MB/145 MB of archives.
After this operation, 488 MB of additional disk space will be used.
Selecting previously unselected package libatk1.0-data.
(Reading database ... 118709 files and directories currently installed.)
Preparing to unpack .../00-libatk1.0-data_2.36.0-3build1_all.deb ...
Unpacking libatk1.0-data (2.36.0-3build1) ...
Selecting previously unselected package libatk1.0-0:amd64.
Preparing to unpack .../01-libatk1.0-0_2.36.0-3b

In [16]:
import time
from selenium import webdriver

def scrape_products(url):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(1)
        products = []
        product_cards = driver.find_elements("class name", "thumbnail")
        for card in product_cards:
            title = card.find_element("class name", "title").get_attribute("title")
            price = card.find_element("class name", "price").text
            description = card.find_element("class name", "description").text
            rating_element = card.find_element("css selector", ".ratings p[data-rating]")
            rating = int(rating_element.get_attribute("data-rating"))
            products.append({"title": title, "price": price, "description": description, "rating": rating})
        return products
    finally:
        driver.quit()

In [17]:

url = "https://webscraper.io/test-sites/e-commerce/static/computers/laptops"
products = scrape_products(url)
try:
    products = scrape_products(url)
    output = ""
    for i, q in enumerate(products):
        output += f"Product #{i}:\n{q}\n"
    print(output)
except Exception as e:
    print(str(e))


Product #0:
{'title': 'Packard 255 G2', 'price': '$416.99', 'description': '15.6", AMD E2-3800 1.3GHz, 4GB, 500GB, Windows 8.1', 'rating': 2}
Product #1:
{'title': 'Aspire E1-510', 'price': '$306.99', 'description': '15.6", Pentium N3520 2.16GHz, 4GB, 500GB, Linux', 'rating': 3}
Product #2:
{'title': 'ThinkPad T540p', 'price': '$1178.99', 'description': '15.6", Core i5-4200M, 4GB, 500GB, Win7 Pro 64bit', 'rating': 1}
Product #3:
{'title': 'ProBook', 'price': '$739.99', 'description': '14", Core i5 2.6GHz, 4GB, 500GB, Win7 Pro 64bit', 'rating': 4}
Product #4:
{'title': 'ThinkPad X240', 'price': '$1311.99', 'description': '12.5", Core i5-4300U, 8GB, 240GB SSD, Win7 Pro 64bit', 'rating': 3}
Product #5:
{'title': 'Aspire E1-572G', 'price': '$581.99', 'description': '15.6", Core i5-4200U, 8GB, 1TB, Radeon R7 M265, Windows 8.1', 'rating': 1}



In [18]:
def scrape_product_detail(url, delay=2.0):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(delay)
        caption_headings = driver.find_elements("css selector", ".caption h4")
        price = caption_headings[0].text
        title = caption_headings[1].text
        description = driver.find_element("css selector", ".description").text
        star_elements = driver.find_elements("css selector", ".ratings [class~='ws-icon-star']")
        rating = len(star_elements)
        return {"title": title, "price": price, "description": description, "rating": rating}
    finally:
        driver.quit()

In [19]:

url = "https://webscraper.io/test-sites/e-commerce/static/product/32"
detail = scrape_product_detail(url)
print(f"Product details: {detail}")

Product details: {'title': 'Aspire E1-510', 'price': '$306.99', 'description': '15.6", Pentium N3520 2.16GHz, 4GB, 500GB, Linux', 'rating': 3}


In [20]:
def scroll_and_scrape(url, scroll_pause=2.0):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        last_height = driver.execute_script("return document.body.scrollHeight")
        poll_interval = 0.2
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            elapsed = 0.0
            new_height = last_height
            while elapsed < scroll_pause:
                time.sleep(poll_interval)
                elapsed += poll_interval
                new_height = driver.execute_script("return document.body.scrollHeight")
                if new_height != last_height:
                    break
            if new_height == last_height:
                break
            last_height = new_height
        products = []
        seen = set()
        product_cards = driver.find_elements("class name", "thumbnail")
        for card in product_cards:
            title = card.find_element("class name", "title").get_attribute("title")
            price = card.find_element("class name", "price").text
            description = card.find_element("class name", "description").text
            star_elements = card.find_elements("css selector", ".ratings [class~='ws-icon-star']")
            rating = len(star_elements)
            key = (title, price)
            if key in seen:
                continue
            seen.add(key)
            products.append({"title": title, "price": price, "description": description, "rating": rating})
        return products
    finally:
        driver.quit()

In [21]:
url = "https://webscraper.io/test-sites/e-commerce/scroll/computers/laptops"
try:
    products = scroll_and_scrape(url)
    output = ""
    for i, q in enumerate(products):
        output += f"Product #{i}:\n{q}\n"
    print(output)
except Exception as e:
    print(str(e))

Product #0:
{'title': 'Asus VivoBook X441NA-GA190', 'price': '$295.99', 'description': 'Asus VivoBook X441NA-GA190 Chocolate Black, 14", Celeron N3450, 4GB, 128GB SSD, Endless OS, ENG kbd', 'rating': 5}
Product #1:
{'title': 'Prestigio SmartBook 133S Dark Grey', 'price': '$299', 'description': 'Prestigio SmartBook 133S Dark Grey, 13.3" FHD IPS, Celeron N3350 1.1GHz, 4GB, 32GB, Windows 10 Pro + Office 365 1 gadam', 'rating': 5}
Product #2:
{'title': 'Prestigio SmartBook 133S Gold', 'price': '$299', 'description': 'Prestigio SmartBook 133S Gold, 13.3" FHD IPS, Celeron N3350 1.1GHz, 4GB, 32GB, Windows 10 Pro + Office 365 1 gadam', 'rating': 5}
Product #3:
{'title': 'Aspire E1-510', 'price': '$306.99', 'description': '15.6", Pentium N3520 2.16GHz, 4GB, 500GB, Linux', 'rating': 5}
Product #4:
{'title': 'Lenovo V110-15IAP', 'price': '$321.94', 'description': 'Lenovo V110-15IAP, 15.6" HD, Celeron N3350 1.1GHz, 4GB, 128GB SSD, Windows 10 Home', 'rating': 5}
Product #5:
{'title': 'Lenovo V110-1